In [1]:
# I'm working with the DvDrental data set to practice pandas merges which i learned in the bootcamp, but wnated to reinforce with real data
import pandas as pd

In [2]:
category = pd.read_csv('category.csv')
film_category = pd.read_csv('film_category.csv')
film = pd.read_csv('film.csv')
inventory = pd.read_csv('inventory.csv')
rental = pd.read_csv('rental.csv')

### copied SQL block to pandas to use as a reference for merges.
```
from film f
left join inventory i on i.film_id = f.film_id
left join rental r on r.inventory_id = i.inventory_id 
join film_category fc on fc.film_id = f.film_id 
join category c on c.category_id = fc.category_id 
```

In [3]:
print(film.columns)
print(inventory.columns)

Index(['film_id', 'title', 'description', 'release_year', 'language_id',
       'rental_duration', 'rental_rate', 'length', 'replacement_cost',
       'rating', 'last_update', 'special_features', 'fulltext'],
      dtype='str')
Index(['inventory_id', 'film_id', 'store_id', 'last_update'], dtype='str')


In [4]:
df = pd.merge(film, inventory, on = 'film_id', how = 'left')

In [5]:
print(rental.columns)

Index(['rental_id', 'rental_date', 'inventory_id', 'customer_id',
       'return_date', 'staff_id', 'last_update'],
      dtype='str')


In [6]:
df = pd.merge(df,rental, on = 'inventory_id', how = 'left')

In [7]:
print(film_category.columns)

Index(['film_id', 'category_id', 'last_update'], dtype='str')


In [8]:
# Last update column not needed for any of my analysis, so i'm dropping it from film_category and category to make the merge easier.

film_category = film_category[['film_id', 'category_id']]

In [9]:
df = pd.merge(df,film_category, on = 'film_id', how = 'left')

In [10]:
print(category.columns)

Index(['category_id', 'name', 'last_update'], dtype='str')


In [11]:
category.rename(columns={'name':'genre'}, inplace = True)
category = category[['category_id','genre']]

In [12]:
df = pd.merge(df,category, on = 'category_id', how = 'left')

In [13]:
df.columns

Index(['film_id', 'title', 'description', 'release_year', 'language_id',
       'rental_duration', 'rental_rate', 'length', 'replacement_cost',
       'rating', 'last_update_x', 'special_features', 'fulltext',
       'inventory_id', 'store_id', 'last_update_y', 'rental_id', 'rental_date',
       'customer_id', 'return_date', 'staff_id', 'last_update', 'category_id',
       'genre'],
      dtype='str')

In [14]:
data = df #backup of the merged data

In [15]:
# dropping columns i don't expect to use
df = df[['film_id','title','rental_rate','rating','inventory_id','rental_id','rental_date','return_date','genre']]

In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16087 entries, 0 to 16086
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   film_id       16087 non-null  int64  
 1   title         16087 non-null  str    
 2   rental_rate   16087 non-null  float64
 3   rating        16087 non-null  str    
 4   inventory_id  16045 non-null  float64
 5   rental_id     16044 non-null  float64
 6   rental_date   16044 non-null  str    
 7   return_date   15861 non-null  str    
 8   genre         16087 non-null  str    
dtypes: float64(3), int64(1), str(5)
memory usage: 2.2 MB


In [17]:
df['rental_date'] = pd.to_datetime(df['rental_date'])
df['return_date'] = pd.to_datetime(df['return_date'])

In [18]:
#showing the 42 films with 0 inventory copies.
no_inventory = df[df['inventory_id'].isnull()].reset_index(drop = True)
display(no_inventory)

,film_id,title,rental_rate,rating,inventory_id,rental_id,rental_date,return_date,genre
0,14,Alice Fantasia,0.99,NC-17,NaN,NaN,NaT,NaT,Classics
1,33,Apollo Teen,2.99,PG-13,NaN,NaN,NaT,NaT,Drama
2,36,Argonauts Town,0.99,PG-13,NaN,NaN,NaT,NaT,Animation
3,38,Ark Ridgemont,0.99,NC-17,NaN,NaN,NaT,NaT,Action
4,41,Arsenic Independence,0.99,PG,NaN,NaN,NaT,NaT,Travel
5,87,Boondock Ballroom,0.99,NC-17,NaN,NaN,NaT,NaT,Travel
6,108,Butch Panther,0.99,PG-13,NaN,NaN,NaT,NaT,New
7,128,Catch Amistad,0.99,G,NaN,NaN,NaT,NaT,Foreign
8,144,Chinatown Gladiator,4.99,PG,NaN,NaN,NaT,NaT,New
9,148,Chocolate Duck,2.99,R,NaN,NaN,NaT,NaT,Foreign


In [19]:
# now dropping those films with no inventory from our data
df = df.dropna(subset = 'inventory_id')

In [20]:
df.info()

<class 'pandas.DataFrame'>
Index: 16045 entries, 0 to 16086
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   film_id       16045 non-null  int64         
 1   title         16045 non-null  str           
 2   rental_rate   16045 non-null  float64       
 3   rating        16045 non-null  str           
 4   inventory_id  16045 non-null  float64       
 5   rental_id     16044 non-null  float64       
 6   rental_date   16044 non-null  datetime64[us]
 7   return_date   15861 non-null  datetime64[us]
 8   genre         16045 non-null  str           
dtypes: datetime64[us](2), float64(3), int64(1), str(3)
memory usage: 1.6 MB


In [21]:
# one film has no rental id?  let's see
df[df['rental_id'].isnull()]

,film_id,title,rental_rate,rating,inventory_id,rental_id,rental_date,return_date,genre
64,1,Academy Dinosaur,0.99,PG,5.0,NaN,NaT,NaT,Documentary


In [22]:
# not a dead film, just one of the 8 copies (copy # 5) was somehow never rented, the other 7 copies all were rented at least twice.
df[df['film_id'] == 1]

,film_id,title,rental_rate,rating,inventory_id,rental_id,rental_date,return_date,genre
52,1,Academy Dinosaur,0.99,PG,1.0,4863.0,2005-07-08 19:03:15,2005-07-11 21:29:15,Documentary
53,1,Academy Dinosaur,0.99,PG,1.0,11433.0,2005-08-02 20:13:10,2005-08-11 21:35:10,Documentary
54,1,Academy Dinosaur,0.99,PG,1.0,14714.0,2005-08-21 21:27:43,2005-08-30 22:26:43,Documentary
55,1,Academy Dinosaur,0.99,PG,2.0,972.0,2005-05-30 20:21:07,2005-06-06 00:36:07,Documentary
56,1,Academy Dinosaur,0.99,PG,2.0,2117.0,2005-06-17 20:24:00,2005-06-23 17:45:00,Documentary
57,1,Academy Dinosaur,0.99,PG,2.0,4187.0,2005-07-07 10:41:31,2005-07-11 06:25:31,Documentary
58,1,Academy Dinosaur,0.99,PG,2.0,9449.0,2005-07-30 22:02:34,2005-08-06 02:09:34,Documentary
59,1,Academy Dinosaur,0.99,PG,2.0,15453.0,2005-08-23 01:01:01,2005-08-30 20:08:01,Documentary
60,1,Academy Dinosaur,0.99,PG,3.0,10126.0,2005-07-31 21:36:07,2005-08-03 23:59:07,Documentary
61,1,Academy Dinosaur,0.99,PG,3.0,15421.0,2005-08-22 23:56:37,2005-08-25 18:58:37,Documentary


In [23]:
df['rental_id'] = df['rental_id'].fillna(0).astype('Int64')
df['inventory_id'] = df['inventory_id'].astype('Int64')

In [24]:
df.info()

<class 'pandas.DataFrame'>
Index: 16045 entries, 0 to 16086
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   film_id       16045 non-null  int64         
 1   title         16045 non-null  str           
 2   rental_rate   16045 non-null  float64       
 3   rating        16045 non-null  str           
 4   inventory_id  16045 non-null  Int64         
 5   rental_id     16045 non-null  Int64         
 6   rental_date   16044 non-null  datetime64[us]
 7   return_date   15861 non-null  datetime64[us]
 8   genre         16045 non-null  str           
dtypes: Int64(2), datetime64[us](2), float64(1), int64(1), str(3)
memory usage: 1.6 MB


In [25]:
df = df.sort_values(['film_id','inventory_id']).reset_index(drop = True)
df

,film_id,title,rental_rate,rating,inventory_id,rental_id,rental_date,return_date,genre
0,1,Academy Dinosaur,0.99,PG,1,4863,2005-07-08 19:03:15,2005-07-11 21:29:15,Documentary
1,1,Academy Dinosaur,0.99,PG,1,11433,2005-08-02 20:13:10,2005-08-11 21:35:10,Documentary
2,1,Academy Dinosaur,0.99,PG,1,14714,2005-08-21 21:27:43,2005-08-30 22:26:43,Documentary
3,1,Academy Dinosaur,0.99,PG,2,972,2005-05-30 20:21:07,2005-06-06 00:36:07,Documentary
4,1,Academy Dinosaur,0.99,PG,2,2117,2005-06-17 20:24:00,2005-06-23 17:45:00,Documentary
...,...,...,...,...,...,...,...,...,...
16040,1000,Zorro Ark,4.99,NC-17,4581,711,2005-05-29 03:49:03,2005-05-31 08:29:03,Comedy
16041,1000,Zorro Ark,4.99,NC-17,4581,1493,2005-06-15 21:50:32,2005-06-17 01:02:32,Comedy
16042,1000,Zorro Ark,4.99,NC-17,4581,6712,2005-07-12 13:24:47,2005-07-20 09:35:47,Comedy
16043,1000,Zorro Ark,4.99,NC-17,4581,9701,2005-07-31 07:32:21,2005-08-01 05:07:21,Comedy


In [26]:
df['previous_return'] = df.groupby('inventory_id')['return_date'].shift(1)

In [27]:
df['days_shelved'] = df['rental_date'].dt.date - df['previous_return'].dt.date
df['days_shelved'] = pd.to_timedelta(df['days_shelved']).dt.days.astype('Int64')
df.head()

,film_id,title,rental_rate,rating,inventory_id,rental_id,rental_date,return_date,genre,previous_return,days_shelved
0,1,Academy Dinosaur,0.99,PG,1,4863,2005-07-08 19:03:15,2005-07-11 21:29:15,Documentary,NaT,<NA>
1,1,Academy Dinosaur,0.99,PG,1,11433,2005-08-02 20:13:10,2005-08-11 21:35:10,Documentary,2005-07-11 21:29:15,22
2,1,Academy Dinosaur,0.99,PG,1,14714,2005-08-21 21:27:43,2005-08-30 22:26:43,Documentary,2005-08-11 21:35:10,10
3,1,Academy Dinosaur,0.99,PG,2,972,2005-05-30 20:21:07,2005-06-06 00:36:07,Documentary,NaT,<NA>
4,1,Academy Dinosaur,0.99,PG,2,2117,2005-06-17 20:24:00,2005-06-23 17:45:00,Documentary,2005-06-06 00:36:07,11


In [47]:
slow_movers = df.groupby(['film_id','title','rating','genre'])['days_shelved'].agg(['mean','max']).rename(columns = {'mean':'average_shelftime','max':'max_shelftime'}).sort_values('max_shelftime', ascending = False).reset_index()
slow_movers = slow_movers[slow_movers['max_shelftime'] >= 46]
display(slow_movers)

,film_id,title,rating,genre,average_shelftime,max_shelftime
0,39,Armageddon Lost,G,Sci-Fi,29.857143,201
1,476,Jason Trap,NC-17,Family,26.647059,201
2,388,Gunfight Moon,NC-17,Comedy,35.222222,200
3,67,Berets Agent,PG-13,Action,27.8,200
4,254,Driver Annie,PG-13,Sports,30.615385,200
...,...,...,...,...,...,...
163,998,Zhivago Core,NC-17,Horror,40.428571,188
164,936,Vanishing Rocky,NC-17,Music,31.416667,188
165,183,Conversation Downhill,R,Family,31.0,188
166,191,Crooked Frogmen,PG-13,Children,26.529412,187


In [48]:
slow_movers.describe()

,film_id,average_shelftime,max_shelftime
count,168.000000,168.0,168.0
mean,488.994048,32.624304,193.994048
std,299.531400,8.825593,3.215786
min,2.000000,22.708333,187.0
25%,227.750000,26.90625,192.0
50%,465.500000,29.928571,194.0
75%,777.500000,34.222222,196.0
max,998.000000,62.25,201.0


In [ ]:
# while these 168 slow moving films at least one copy of each has a max shelf time between 187 and 201 days, 
# looking at the averages per film, we can see that shelftime average sits between 22 days and 62 days 
# per film.